In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

#Define system-agnostic base directory (current-working-directory)
BASE_DIR = Path.cwd()

train_path = BASE_DIR / "churn_bigml_80.csv"
test_path = BASE_DIR / "churn_bigml_20.csv"
print(f"Looking for training data at {train_path}")
print(f"Looking for testing data at {test_path}")


Looking for training data at /home/john123/ML_projects/Level2/Task1/churn_bigml_80.csv
Looking for testing data at /home/john123/ML_projects/Level2/Task1/churn_bigml_20.csv


In [4]:
#Load data using Path ojects directly
if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError("Verify your CSV files are inside current directory")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")
print("\nFirst 3 rows of training set:")
print(train_df.head(3))

Train shape: (2666, 20) | Test shape: (667, 20)

First 3 rows of training set:
  State  Account length  Area code International plan Voice mail plan  \
0    KS             128        415                 No             Yes   
1    OH             107        415                 No             Yes   
2    NJ             137        415                 No              No   

   Number vmail messages  Total day minutes  Total day calls  \
0                     25              265.1              110   
1                     26              161.6              123   
2                      0              243.4              114   

   Total day charge  Total eve minutes  Total eve calls  Total eve charge  \
0             45.07              197.4               99             16.78   
1             27.47              195.5              103             16.62   
2             41.38              121.2              110             10.30   

   Total night minutes  Total night calls  Total night charge 

In [7]:
target_col = "Churn"
#Separate features and target
X_train = train_df.drop(columns=[target_col, "State"])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col, "State"])
y_test = test_df[target_col]
#Find numeric features requiring scaling
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()

#Scale arrays explicitly
scaler = StandardScaler()

#Fit and tranform on training data, but only transform for testing data
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

# One-hot encode categorical columns
categorical_features = X_train.select_dtypes(include=["object", "bool", "str"]).columns.tolist()

X_train_scaled = pd.get_dummies(X_train_scaled, columns=categorical_features, drop_first=True)
X_test_scaled = pd.get_dummies(X_test_scaled, columns=categorical_features, drop_first=True)

# Align test columns with train (get_dummies can create different columns)
X_test_scaled = X_test_scaled.reindex(columns=X_train_scaled.columns, fill_value=0)

print("==== Data successfully scaled with zero test set leakage ====")

==== Data successfully scaled with zero test set leakage ====


In [8]:
#Fit the model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

#Extract the math parameters
coefficients = model.coef_[0]
intercept = model.intercept_[0]

print(f"Model Intercept (Base Log-Odds): {intercept:.4f}\n")
print(f"{'Feature':<25} | {'Log-Odds Coef':<15} | {'Odds Ratio(e^w)':<15}")
print("-" * 60)

for col, coef in zip(X_train_scaled.columns, coefficients):
    odds_ratio = np.exp(coef)
    print(f"{col:<25} | {coef:<15.4f} | {odds_ratio:<15.4f}")

Model Intercept (Base Log-Odds): -2.1012

Feature                   | Log-Odds Coef   | Odds Ratio(e^w)
------------------------------------------------------------
Account length            | 0.0348          | 1.0354         
Area code                 | -0.0255         | 0.9748         
Number vmail messages     | 0.2502          | 1.2843         
Total day minutes         | 0.3382          | 1.4024         
Total day calls           | 0.0561          | 1.0577         
Total day charge          | 0.3385          | 1.4028         
Total eve minutes         | 0.1441          | 1.1549         
Total eve calls           | -0.0166         | 0.9836         
Total eve charge          | 0.1409          | 1.1514         
Total night minutes       | 0.0703          | 1.0728         
Total night calls         | 0.0372          | 1.0379         
Total night charge        | 0.0702          | 1.0727         
Total intl minutes        | 0.1321          | 1.1413         
Total intl calls          | -